In [1]:
import kagglehub
import pandas as pd
import os

path = kagglehub.dataset_download("pavansubhasht/ibm-hr-analytics-attrition-dataset")
print("Fichiers disponibles :", os.listdir(path))

df = pd.read_csv(os.path.join(path, "WA_Fn-UseC_-HR-Employee-Attrition.csv"))

X = df.drop("Attrition", axis=1)
y = df["Attrition"]

print("Forme :", df.shape)
df.head()

c:\Users\Math34\Mathieu\Projets Ipssi\Ipssi_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fichiers disponibles : ['WA_Fn-UseC_-HR-Employee-Attrition.csv']
Forme : (1470, 35)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [8]:
from sklearn.model_selection import train_test_split
def split_train_val_test(X, y, test_size=0.2, val_size=0.2, random_state=42):
    """Découpe X, y en trois jeux : train, validation, test.
    Doit renvoyer 6 objets : X_train, X_val, X_test, y_train, y_val, y_test.
    Les proportions doivent rester respectées (utiliser stratify=y).
    """

    X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size = val_size, random_state=random_state, stratify=y_trainval)

    return X_train, X_val, X_test, y_train, y_val, y_test

    # TODO : premier split pour isoler le test (test_size)
    # TODO : deuxième split sur le reste pour isoler la validation
    # TODO : penser à recalculer la proportion val sur le reste
    # TODO : renvoyer les 6 objets
    # stratify=y : garder la même proportion de classes dans chaque jeu
    # qu'au départ, pour ne pas créer un test sans la classe rare
    pass

X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(X, y)

#cas normal
print(len(X))
print(len(X_train + X_val + X_test))
print(len(y_train + y_val + y_test))

#cas limite 
#la focntion plante proprement
#X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(X, y, val_size=0)

#cas adversarial 
print(y.value_counts(normalize=True).round(2))
print(y_train.value_counts(normalize=True).round(2))
print(y_val.value_counts(normalize=True).round(2))
print(y_test.value_counts(normalize=True).round(2))




1470
1470
1470
Attrition
No     0.84
Yes    0.16
Name: proportion, dtype: float64
Attrition
No     0.84
Yes    0.16
Name: proportion, dtype: float64
Attrition
No     0.84
Yes    0.16
Name: proportion, dtype: float64
Attrition
No     0.84
Yes    0.16
Name: proportion, dtype: float64


In [ ]:
X.iloc[1]

Age                                             49
BusinessTravel                   Travel_Frequently
DailyRate                                      279
Department                  Research & Development
DistanceFromHome                                 8
Education                                        1
EducationField                       Life Sciences
EmployeeCount                                    1
EmployeeNumber                                   2
EnvironmentSatisfaction                          3
Gender                                        Male
HourlyRate                                      61
JobInvolvement                                   2
JobLevel                                         2
JobRole                         Research Scientist
JobSatisfaction                                  2
MaritalStatus                              Married
MonthlyIncome                                 5130
MonthlyRate                                  24907
NumCompaniesWorked             

In [4]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier

def bootstrap_scores(modele, X, y, n_iterations=30, random_state=42):
    """Évalue la stabilité d'un modèle par bootstrap.
    Pour chaque itération : tirer un échantillon AVEC REMISE de même taille
    que X, entraîner, évaluer sur les points NON tirés (out-of-bag).
    Doit renvoyer la liste des scores et afficher moyenne et écart-type.
    """
    X = pd.get_dummies(X)
    indices = set(range(len(X)))
    scores = []
    rng = np.random.default_rng(random_state)
    for i in range(n_iterations) :
        indices_train = rng.choice(len(X), size=len(X), replace=True)
        OOB = list(set(range(len(X))) - set(indices_train))
        if len(OOB) == 0 : 
            continue
        else :
            modele.fit(X.iloc[indices_train], y.iloc[indices_train])
            score = modele.score(X.iloc[OOB], y.iloc[OOB])
            scores.append(score)
            
    print(f"Score moyen sur {n_iterations} bootstraps : {np.mean(scores):.3f} (± {np.std(scores):.3f})")
    return scores
    

    # TODO : rng = np.random.default_rng(random_state)
    # TODO : pour chaque itération, tirer des indices avec remise (rng.choice avec replace=True)
    # TODO : identifier les indices out-of-bag (OOB) pour évaluer
    #    OOB = les points jamais tirés dans cet échantillon bootstrap,
    #    ils n'ont pas servi à l'entraînement : jeu de test gratuit
    # TODO : entraîner sur l'échantillon, scorer sur l'out-of-bag
    # TODO : renvoyer la liste, afficher moyenne ± écart-type
    pass

modele = RandomForestClassifier(n_estimators=100, random_state=42)
bootstrap_scores(modele, X, y)

Score moyen sur 30 bootstraps : 0.860 (± 0.013)


[0.8401486988847584,
 0.8637200736648251,
 0.8552875695732839,
 0.8462962962962963,
 0.8706739526411658,
 0.8692449355432781,
 0.8704761904761905,
 0.871939736346516,
 0.8518518518518519,
 0.8330404217926186,
 0.8701298701298701,
 0.8757062146892656,
 0.8543516873889876,
 0.8643122676579925,
 0.8632958801498127,
 0.8768382352941176,
 0.8628884826325411,
 0.8658088235294118,
 0.8651685393258427,
 0.8579335793357934,
 0.8606403013182674,
 0.8732943469785575,
 0.8612167300380228,
 0.872865275142315,
 0.8324022346368715,
 0.8687615526802218,
 0.8612167300380228,
 0.847985347985348,
 0.8376383763837638,
 0.8402903811252269]

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import LeaveOneOut
import time
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
import numpy as np

def evaluer_en_cross_val(modele, X, y, cv=5):
    """Lance une validation croisée k-fold et résume les résultats.
    Doit afficher les k scores, leur moyenne et leur écart-type.
    Un grand écart-type entre folds = modèle instable, à signaler.
    """

    scores = cross_val_score(modele, X, y, cv=cv, scoring="accuracy")
    print("les k scores :", scores.round(2))

    #variation des scores de plus ou mois un pourcent autour de la moyenne : très stable
    print(f"moyenne : {np.mean(scores).round(2)}, écart-type : {np.std(scores).round(2)}")
    


    # TODO : scores = cross_val_score(modele, X, y, cv=k, scoring="accuracy")
    # TODO : afficher les k scores
    # TODO : afficher moyenne et écart-type, et commenter la stabilité
    pass

modele = RandomForestClassifier(n_estimators=10, random_state=42)
X = pd.get_dummies(X)

evaluer_en_cross_val(modele, X, y, cv=5)

#cas limite
# t0 = time.perf_counter()
# evaluer_en_cross_val(modele, X, y, cv=LeaveOneOut())
# t1 = time.perf_counter()
# print("temps d'execution (en s): ", t1.round(2))

#Cas adversarial 
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
df = pd.read_csv(os.path.join(path, "creditcard.csv"))
df = df.sample(10000, random_state=42)
X_credit = df.drop("Class", axis=1)
y_credit = df["Class"]

evaluer_en_cross_val(modele, X_credit, y_credit, cv=5)
evaluer_en_cross_val(modele, X_credit, y_credit, cv=StratifiedKFold(5))
#New dataset : StratifiedKFold ne change rien ici car le problème est la métrique (fraude : 0,17 pourcent)

evaluer_en_cross_val(modele, X, y, cv=5)
evaluer_en_cross_val(modele, X, y, cv=StratifiedKFold(5))
#avec mon dataset resultats similaires avec StratifiedKFold surement parce que mon dataset n'est pas assez déséquilibré (le hasard suffit pour la repartion entre les folds)


les k scores : [0.85 0.87 0.85 0.87 0.85]
moyenne : 0.86, écart-type : 0.01
les k scores : [1. 1. 1. 1. 1.]
moyenne : 1.0, écart-type : 0.0
les k scores : [1. 1. 1. 1. 1.]
moyenne : 1.0, écart-type : 0.0
les k scores : [0.85 0.87 0.85 0.87 0.85]
moyenne : 0.86, écart-type : 0.01
les k scores : [0.85 0.87 0.85 0.87 0.85]
moyenne : 0.86, écart-type : 0.01


In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
from sklearn.metrics import accuracy_score



def rapport_metier(y_true, y_pred, cout_fn=10, cout_fp=1):
    """Affiche la matrice de confusion et les métriques, puis calcule un
    COÛT MÉTIER total : (nb faux négatifs * cout_fn) + (nb faux positifs * cout_fp).
    Doit afficher precision, recall, F1, et le coût total.
    Permet de comparer deux modèles sur l'argent perdu, pas sur l'accuracy.
    """
    matrice = confusion_matrix(y_true, y_pred)
    VN, FP, FN, VP = matrice.ravel()
    
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall    = recall_score(y_true, y_pred)
    f1        = f1_score(y_true, y_pred)
    cout_total = FN * cout_fn + FP * cout_fp
    
    print("precision=" , precision, "recall=", recall, "f1=", f1, "cout métier=", cout_total)

    
    # TODO : matrice = confusion_matrix(y_true, y_pred) -> en extraire VN, FP, FN, VP
    # TODO : afficher precision, recall, f1
    # TODO : cout_total = FN * cout_fn + FP * cout_fp
    # TODO : afficher le coût métier total
    return cout_total

path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
df = pd.read_csv(os.path.join(path, "creditcard.csv"))
df = df.sample(160000, random_state=42)
X_credit = df.drop("Class", axis=1)
y_credit = df["Class"]
X_credit = pd.get_dummies(X_credit)
X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(X_credit, y_credit)

modele_A = RandomForestClassifier(n_estimators=70, random_state=42)
modele_A.fit(X_train, y_train)
y_pred_A = modele_A.predict(X_test)
modele_B = RandomForestClassifier(n_estimators=70, class_weight="balanced", random_state=42)
modele_B.fit(X_train, y_train)
y_pred_B = modele_B.predict(X_test)

print("Rapport modele A :" , rapport_metier(y_test, y_pred_A))
print("Rapport modele B :" , rapport_metier(y_test, y_pred_B))
#tro peu d'arbres pour que balanced soit efficace ? : le cout metier est superieur avec le modele B qu'avec le A

#cas limite 
y_pred_paresseux = np.zeros(len(y_test))
accuracy = accuracy_score(y_test, y_pred_paresseux)
                          
print("Accuracy =", accuracy)
rapport_metier(y_test, y_pred_paresseux)
#recall à 0 malgrès les 99% d'accuracy

#cas adversarial
print(confusion_matrix(y_test, y_pred_A))
print(confusion_matrix(y_test, y_pred_paresseux))
#la c'est un desastre le modele laisse passer toutes les fraudes malgrès une très bonne accuracy. 

precision= 0.975609756097561 recall= 0.7547169811320755 f1= 0.851063829787234 cout métier= 131
Rapport modele A : 131
precision= 0.9 recall= 0.6792452830188679 f1= 0.7741935483870968 cout métier= 174
Rapport modele B : 174
Accuracy = 0.99834375
precision= 0.0 recall= 0.0 f1= 0.0 cout métier= 530
[[31946     1]
 [   13    40]]
[[31947     0]
 [   53     0]]
